In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

In [3]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train:", train.shape)
print("Test :", test.shape)

Train: (690088, 15)
Test : (295753, 14)


In [4]:
X = train.drop(columns=["health_condition"])
y = train["health_condition"]

X_test = test.copy()

In [5]:
cat_features = X.select_dtypes(include="object").columns.tolist()

print(cat_features)

['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


In [6]:
# Identify categorical columns
cat_features = X.select_dtypes(include="object").columns.tolist()

# Fill missing categorical values
for col in cat_features:
    X[col] = X[col].fillna("Unknown").astype(str)
    X_test[col] = X_test[col].fillna("Unknown").astype(str)

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_valid.shape)

(552070, 14)
(138018, 14)


In [8]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=100
)

In [9]:
model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 0.9375496	test: 0.9392108	best: 0.9392108 (0)	total: 921ms	remaining: 15m 20s
100:	learn: 0.9657833	test: 0.9662725	best: 0.9663087 (99)	total: 1m 10s	remaining: 10m 28s
200:	learn: 0.9665350	test: 0.9663740	best: 0.9664174 (189)	total: 2m 19s	remaining: 9m 14s
300:	learn: 0.9671092	test: 0.9666348	best: 0.9666348 (300)	total: 3m 29s	remaining: 8m 6s
400:	learn: 0.9676146	test: 0.9666638	best: 0.9667145 (317)	total: 4m 40s	remaining: 6m 58s
500:	learn: 0.9682087	test: 0.9665551	best: 0.9667145 (317)	total: 5m 51s	remaining: 5m 50s
600:	learn: 0.9687014	test: 0.9667145	best: 0.9667362 (582)	total: 7m 1s	remaining: 4m 39s
700:	learn: 0.9693010	test: 0.9667724	best: 0.9667869 (699)	total: 8m 11s	remaining: 3m 29s
800:	learn: 0.9697339	test: 0.9668232	best: 0.9668956 (763)	total: 9m 23s	remaining: 2m 19s
900:	learn: 0.9701252	test: 0.9669246	best: 0.9669246 (899)	total: 10m 36s	remaining: 1m 9s
999:	learn: 0.9707247	test: 0.9670043	best: 0.9670260 (991)	total: 11m 48s	remaining: 

CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=1000, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

#### Balanced Accuracy

In [10]:
from sklearn.metrics import balanced_accuracy_score

pred = model.predict(X_valid)

score = balanced_accuracy_score(y_valid, pred)

print("Balanced Accuracy:", score)

Balanced Accuracy: 0.8738833824285738


# Results

Model: CatBoostClassifier

Balanced Accuracy: 0.87388

Observations:
- Good baseline performance.
- Model handles categorical features and missing values well.
- There is still room for improvement through feature engineering and hyperparameter tuning.